# OfferSense Marketing Analytics - EDA
## Exploratory Data Analysis & Campaign Performance Insights

This notebook provides comprehensive analysis of marketing campaign data including statistical summaries, visualizations, and actionable insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Data from Database

In [ ]:
# Connect to database and load data
db_path = Path('../offersense.db')

if not db_path.exists():
    print("⚠️  Database not found. Make sure to run the backend first.")
    print("Creating sample data for demonstration...")
    data = {
        'name': ['Campaign A', 'Campaign B', 'Campaign C', 'Campaign D', 'Campaign E'],
        'impressions': [1000, 1500, 2000, 1200, 2500],
        'clicks': [100, 150, 250, 150, 300],
        'conversions': [10, 20, 30, 18, 40]
    }
    df = pd.DataFrame(data)
else:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM campaigns", conn)
    conn.close()

print(f"✓ Data loaded: {len(df)} campaigns")
df.head()

## 2. Data Overview & Quality Check

In [ ]:
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("Data Types:")
print(df.dtypes)
print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("Statistical Summary:")
print(df.describe())

## 3. Calculate Key Performance Metrics

In [ ]:
# Calculate metrics for each campaign
df['CTR'] = (df['clicks'] / df['impressions'] * 100).round(2)
df['Conversion_Rate'] = (df['conversions'] / df['clicks'] * 100).round(2)
df['Drop_off_Rate'] = ((df['clicks'] - df['conversions']) / df['clicks'] * 100).round(2)
df['CPC'] = (df['impressions'] / df['clicks']).round(2)  # Cost Per Click (impressions as proxy)

# Display metrics
print("Campaign Performance Metrics:")
print("="*80)
metrics_df = df[['name', 'impressions', 'clicks', 'conversions', 'CTR', 'Conversion_Rate', 'Drop_off_Rate']]
print(metrics_df.to_string(index=False))

print("\n" + "="*80)
print("Overall KPIs:")
print(f"Total Impressions: {df['impressions'].sum():,}")
print(f"Total Clicks: {df['clicks'].sum():,}")
print(f"Total Conversions: {df['conversions'].sum():,}")
print(f"Overall CTR: {(df['clicks'].sum() / df['impressions'].sum() * 100):.2f}%")
print(f"Overall Conversion Rate: {(df['conversions'].sum() / df['clicks'].sum() * 100):.2f}%")

## 4. Data Visualizations

In [ ]:
### 4.1 Campaign Performance Comparison

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Impressions
axes[0, 0].bar(df['name'], df['impressions'], color='steelblue', alpha=0.8)
axes[0, 0].set_title('Impressions by Campaign', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Impressions')
axes[0, 0].tick_params(axis='x', rotation=45)

# Clicks
axes[0, 1].bar(df['name'], df['clicks'], color='orange', alpha=0.8)
axes[0, 1].set_title('Clicks by Campaign', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Clicks')
axes[0, 1].tick_params(axis='x', rotation=45)

# Conversions
axes[1, 0].bar(df['name'], df['conversions'], color='green', alpha=0.8)
axes[1, 0].set_title('Conversions by Campaign', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('Conversions')
axes[1, 0].tick_params(axis='x', rotation=45)

# CTR Comparison
axes[1, 1].bar(df['name'], df['CTR'], color='purple', alpha=0.8)
axes[1, 1].set_title('Click-Through Rate (CTR) %', fontweight='bold', fontsize=12)
axes[1, 1].set_ylabel('CTR %')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
### 4.2 Conversion Rate Analysis

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x >= 12 else 'orange' if x >= 10 else 'red' for x in df['Conversion_Rate']]
ax.bar(df['name'], df['Conversion_Rate'], color=colors, alpha=0.8)
ax.axhline(y=12, color='green', linestyle='--', linewidth=2, label='High Performer (≥12%)')
ax.axhline(y=10, color='orange', linestyle='--', linewidth=2, label='Medium Performer (10-12%)')
ax.set_title('Conversion Rate by Campaign', fontweight='bold', fontsize=14)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_xlabel('Campaign', fontsize=12)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### 4.3 Conversion Funnel Analysis

fig, axes = plt.subplots(1, len(df), figsize=(16, 5))

for idx, row in df.iterrows():
    stages = ['Impressions', 'Clicks', 'Conversions']
    values = [row['impressions'], row['clicks'], row['conversions']]
    colors_funnel = ['#3498db', '#e74c3c', '#2ecc71']
    
    axes[idx].bar(stages, values, color=colors_funnel, alpha=0.8)
    axes[idx].set_title(f"{row['name']} Funnel", fontweight='bold')
    axes[idx].set_ylabel('Count')
    
    # Add percentage labels
    for i, (stage, val) in enumerate(zip(stages, values)):
        pct = (val / values[0] * 100) if values[0] > 0 else 0
        axes[idx].text(i, val + 20, f'{pct:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Campaign Segmentation Analysis

In [ ]:
# Segment campaigns based on conversion rate
def segment_campaign(conversion_rate):
    if conversion_rate >= 12:
        return 'High Performer'
    elif conversion_rate >= 10:
        return 'Medium Performer'
    else:
        return 'Low Performer'

df['Segment'] = df['Conversion_Rate'].apply(segment_campaign)

# Display segmentation
print("Campaign Segmentation:")
print("="*80)
for segment in ['High Performer', 'Medium Performer', 'Low Performer']:
    segment_df = df[df['Segment'] == segment]
    print(f"\n{segment} ({len(segment_df)} campaigns):")
    print(segment_df[['name', 'Conversion_Rate', 'CTR']].to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
segment_counts = df['Segment'].value_counts()
colors = {'High Performer': 'green', 'Medium Performer': 'orange', 'Low Performer': 'red'}
segment_colors = [colors[seg] for seg in segment_counts.index]

ax.pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', 
       colors=segment_colors, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title('Campaign Distribution by Segment', fontweight='bold', fontsize=14)
plt.show()

## 6. Key Insights & Recommendations

In [ ]:
insights = []

# Best performer
best = df.loc[df['Conversion_Rate'].idxmax()]
insights.append(f"🏆 BEST PERFORMER: {best['name']}")
insights.append(f"   - Conversion Rate: {best['Conversion_Rate']}%")
insights.append(f"   - CTR: {best['CTR']}%")
insights.append(f"   - Conversions: {best['conversions']} from {best['clicks']} clicks\n")

# Worst performer
worst = df.loc[df['Conversion_Rate'].idxmin()]
insights.append(f"📉 NEEDS IMPROVEMENT: {worst['name']}")
insights.append(f"   - Conversion Rate: {worst['Conversion_Rate']}%")
insights.append(f"   - CTR: {worst['CTR']}%")
insights.append(f"   - Drop-off Rate: {worst['Drop_off_Rate']}%\n")

# CTR insights
high_ctr = df.loc[df['CTR'].idxmax()]
insights.append(f"📊 HIGHEST ENGAGEMENT: {high_ctr['name']}")
insights.append(f"   - CTR: {high_ctr['CTR']}% (users are clicking ads)\n")

# Volume vs Performance
high_volume = df.loc[df['impressions'].idxmax()]
insights.append(f"📢 HIGHEST VOLUME: {high_volume['name']}")
insights.append(f"   - Impressions: {high_volume['impressions']:,}")
insights.append(f"   - But conversion rate: {high_volume['Conversion_Rate']}%\n")

# Recommendations
insights.append("💡 RECOMMENDATIONS:")
insights.append("1. Scale the best performers - allocate more budget to high-converting campaigns")
insights.append("2. Optimize underperformers - review ad copy, targeting, and landing pages")
insights.append("3. Run A/B tests - compare winning elements across campaigns")
insights.append("4. Monitor funnel drop-off - identify where users are abandoning")
insights.append("5. Adjust bid strategies - focus on cost-effective channels")

for insight in insights:
    print(insight)

## 7. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary_df = df[['name', 'impressions', 'clicks', 'conversions', 'CTR', 'Conversion_Rate', 'Drop_off_Rate', 'Segment']].copy()
summary_df = summary_df.sort_values('Conversion_Rate', ascending=False)

print("\n📊 COMPREHENSIVE CAMPAIGN SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))

print("\n" + "="*100)
print("📈 OVERALL METRICS")
print("="*100)
print(f"{'Total Campaigns':<30} {len(df)}")
print(f"{'Total Impressions':<30} {df['impressions'].sum():,}")
print(f"{'Total Clicks':<30} {df['clicks'].sum():,}")
print(f"{'Total Conversions':<30} {df['conversions'].sum():,}")
print(f"{'Average CTR':<30} {df['CTR'].mean():.2f}%")
print(f"{'Average Conversion Rate':<30} {df['Conversion_Rate'].mean():.2f}%")
print(f"{'Average Drop-off Rate':<30} {df['Drop_off_Rate'].mean():.2f}%")

print("\n" + "="*100)
print("📋 SEGMENTATION SUMMARY")
print("="*100)
for segment in ['High Performer', 'Medium Performer', 'Low Performer']:
    count = len(df[df['Segment'] == segment])
    if count > 0:
        avg_conv = df[df['Segment'] == segment]['Conversion_Rate'].mean()
        print(f"{segment:<25} {count} campaign(s) - Avg Conversion Rate: {avg_conv:.2f}%")